# 03 - Run Judge (run 2): AGE_ persona + assistant judge, categories A/G/E only

Second, separate judge run -- independent of `01_run_judge_run1.ipynb`, no need to
re-run that notebook first (it just reuses the cached raw dataset from
`data/raw/posts.parquet`, downloading it if that cache doesn't exist yet).

Scope for this run:

- **Categories**: only A (Identity), G (Politics), E (Viewpoint) -- see
  `config.AGE_CATEGORIES`.
- **AGE_ persona judge** (toxicity==4, 50 posts/category): taxonomy
  `sycophantic / malicious-manipulative / evil / other` (merges the old
  `malicious` label into `malicious-manipulative`).
- **AGE_ assistant judge** (toxicity==0, 50 posts/category): uses the same
  canonical `AssistantJudgeResult` schema as run 1 -- `is_safe_for_assistant`
  always means "safe content for an assistant agent to post", consistently
  across every run in this repo.

Results are written to `results/AGE_results/AGE_persona.jsonl` and
`results/AGE_results/AGE_assistant_safety.jsonl`, each row also carrying the
post's `content` and `category` so the cache is self-contained for spot-checking.

Safe to interrupt and re-run: sampling is cached to `data/processed/age_*sample.parquet`,
and already-judged post_ids are skipped on every run (see `ResultStore` in
`src/categorizer.py`).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src import config, dataset, prompts
from src.judge_schema import AgePersonaJudgeResult, AssistantJudgeResult
from src.categorizer import ResultStore, get_client, run_judge

## 1. Load posts (cached; downloads only if `data/raw/posts.parquet` is missing)

Category filtering to A/G/E now happens inside `dataset.get_or_create_sample()` via
its `categories` parameter, rather than pre-filtering the dataframe here.

In [2]:
df = dataset.download_posts()
print(f"loaded {len(df)} rows")

loaded 44376 rows


## 2. Stratified samples (50/category, categories A/G/E only)

Cached to `data/processed/age_persona_sample.parquet` and
`data/processed/age_assistant_safety_sample.parquet` -- re-running this cell will
not reshuffle the sample once cached.

In [3]:
age_persona_sample = dataset.get_or_create_sample(
    df,
    name=config.AGE_PERSONA_SAMPLE_NAME,
    toxicity=config.TOXICITY_MALICIOUS,
    per_category=config.AGE_SAMPLES_PER_CATEGORY,
    categories=config.AGE_CATEGORIES,
)
print(f"AGE_ persona sample: {len(age_persona_sample)} rows")
age_persona_sample[config.COL_CATEGORY].value_counts().sort_index()

AGE_ persona sample: 91 rows


topic_label
A    50
E    28
G    13
Name: count, dtype: int64

In [ ]:
age_assistant_sample = dataset.get_or_create_sample(
    df,
    name=config.AGE_ASSISTANT_SAMPLE_NAME,
    toxicity=config.TOXICITY_SAFE,
    per_category=config.AGE_SAMPLES_PER_CATEGORY,
    categories=config.AGE_CATEGORIES,
)
print(f"AGE_ assistant sample: {len(age_assistant_sample)} rows")
age_assistant_sample[config.COL_CATEGORY].value_counts().sort_index()

## 3. Run judges

Each result row also stores `content` and `category` (via `content_key`/`category_key`),
so `results/AGE_results/*.jsonl` is self-contained -- no separate join needed for EDA.
The assistant judge here uses the exact same schema/prompt as run 1's assistant judge.

In [ ]:
client = get_client()

age_persona_store = ResultStore(config.AGE_RESULTS_DIR / f"{config.AGE_PERSONA_JUDGE_NAME}.jsonl")

run_judge(
    client=client,
    store=age_persona_store,
    rows=age_persona_sample.to_dict("records"),
    id_key=config.COL_POST_ID,
    system_prompt=prompts.AGE_PERSONA_JUDGE_SYSTEM,
    build_user_prompt=lambda row: prompts.build_age_persona_prompt(
        post_id=str(row[config.COL_POST_ID]), content=row[config.COL_CONTENT]
    ),
    result_model=AgePersonaJudgeResult,
    content_key=config.COL_CONTENT,
    category_key=config.COL_CATEGORY,
)
print(f"AGE_ persona judge: {len(age_persona_store.seen_ids())} successful rows cached")

In [ ]:
age_assistant_store = ResultStore(config.AGE_RESULTS_DIR / f"{config.AGE_ASSISTANT_JUDGE_NAME}.jsonl")

run_judge(
    client=client,
    store=age_assistant_store,
    rows=age_assistant_sample.to_dict("records"),
    id_key=config.COL_POST_ID,
    system_prompt=prompts.ASSISTANT_JUDGE_SYSTEM,
    build_user_prompt=lambda row: prompts.build_assistant_prompt(
        post_id=str(row[config.COL_POST_ID]), content=row[config.COL_CONTENT]
    ),
    result_model=AssistantJudgeResult,
    content_key=config.COL_CONTENT,
    category_key=config.COL_CATEGORY,
)
print(f"AGE_ assistant judge: {len(age_assistant_store.seen_ids())} successful rows cached")

## 4. Error sanity check

Error rows are retried automatically on the next run of this notebook.

In [ ]:
for name, store in [("AGE_ persona", age_persona_store), ("AGE_ assistant", age_assistant_store)]:
    rows = store.load_all_rows()
    errors = [r for r in rows if "error" in r]
    print(f"{name}: {len(rows)} total rows, {len(errors)} errors, {len(rows) - len(errors)} successful")
    for e in errors[:5]:
        print("  ", e)

## 5. Summary: personas + safety per category

Uses the `category` field stored directly in each result row -- no join needed.

In [ ]:
age_persona_rows = [r for r in age_persona_store.load_all_rows() if "error" not in r]
age_persona_df = pd.DataFrame(age_persona_rows)

persona_by_category = (
    age_persona_df.groupby(["category", "persona"]).size().unstack(fill_value=0)
)
print("Persona counts per category:")
persona_by_category

In [ ]:
age_assistant_rows = [r for r in age_assistant_store.load_all_rows() if "error" not in r]
age_assistant_df = pd.DataFrame(age_assistant_rows)

safety_by_category = (
    age_assistant_df.groupby("category")["is_safe_for_assistant"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "safe_count", "count": "total"})
)
safety_by_category["safe_rate"] = safety_by_category["safe_count"] / safety_by_category["total"]
print("Assistant-safety rate per category:")
safety_by_category